In [ ]:
pip install dwave-ocean-sdk

# Definiendo un BLP

In [ ]:
import dimod

x0 = dimod.Binary("x0")
x1 = dimod.Binary("x1")
x2 = dimod.Binary("x2")

In [ ]:
blp = dimod.ConstrainedQuadraticModel()
blp.set_objective(-5*x0+3*x1-2*x2)
blp.add_constraint(x0 + x2 <= 1, "Restricción 1")
blp.add_constraint(3*x0 -x1 + 3*x2 <= 4, "Restricción 2");

In [ ]:
print("Nuestras variables son:")
print(blp.variables)
print("Nuestra función objetivo es:")
print(blp.objective)
print("Nuestras restricciones son:")
print(blp.constraints)

In [ ]:
sample1 = {"x0":1, "x1":1, "x2":1}
print("La asignación es", sample1)
print("Su coste es", blp.objective.energy(sample1))
print("¿Es factible?",blp.check_feasible(sample1))
print("Las restricciones no cumplidas son")
print(blp.violations(sample1))

In [ ]:
sample2 = {"x0":0, "x1":0, "x2":1}
print("La asignación es", sample2)
print("Su coste es", blp.objective.energy(sample2))
print("¿Es factible?",blp.check_feasible(sample2))
print("Las restricciones no cumplidas son")
print(blp.violations(sample2))

In [ ]:
solver = dimod.ExactCQMSolver()
solution = solver.sample_cqm(blp)
print("La lista de asignaciones es:")
print(solution)

In [ ]:
feasible_sols = solution.filter(lambda s: s.is_feasible)
feasible_sols.first

## Convirtiendo de BLP a QUBO

In [ ]:
y0, y1 = dimod.Binaries(["y0", "y1"])
cqm = dimod.ConstrainedQuadraticModel()
cqm.set_objective(-2*y0-3*y1)
cqm.add_constraint(y0 + 2*y1 <= 2);

In [ ]:
qubo, invert = dimod.cqm_to_bqm(cqm, lagrange_multiplier = 5)
print(qubo)

In [ ]:
solver = dimod.ExactSolver()
result = solver.sample(qubo)
print("Las soluciones son")
print(result)

In [ ]:
samples = []
occurrences = []
for s in result.data():
    samples.append(invert(s.sample))
    occurrences.append(s.num_occurrences)
sampleset = dimod.SampleSet.from_samples_cqm(samples,cqm,
    num_occurrences=occurrences)
print("Las soluciones del problema original son")
print(sampleset)

In [ ]:
final_sols = sampleset.filter(lambda s: s.is_feasible)
final_sols = final_sols.aggregate()
print("Las soluciones finales son")
print(final_sols)